<a href="https://colab.research.google.com/github/htsnet/crewai_agents/blob/main/CrewAI_curso_Alura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Curso ALURA do Youtube**

https://www.youtube.com/watch?v=dRVCO4zFb0k

Busca de informações sobre um assunto e a geração de um texto para publicação no LinkedIn

13/01/2025

Instalação dos pacotes da CrewAI

In [1]:
!pip install -q crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.9/211.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.9/514.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.4/211.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 33.

Importação dos pacotes e chaves

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from google.colab import userdata

# OpenAI
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'

# SERPER
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')


In [4]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

Criação dos Agentes

1 - Buscar de conteúdo na internet

2 - Redator de conteúdo

3 - Editor de conteúdo (correção de erros, ajuste de linguagem, etc.)

In [5]:
buscador = Agent(
    role='Buscador de conteúdo na internet',
    goal='Busque conteúdo online e recente sobre {tema}',
    backstory='Como pesquisador de conteúdo na web, você é responsável por encontrar, filtrar e organizar informações disponíveis na internet de forma eficiente e estratégica. Usando ferramentas de busca, técnicas de análise de dados e boas práticas de pesquisa online, você identifica conteúdos relevantes, confiáveis e de alta qualidade. Seu trabalho pode envolver coletar dados para projetos acadêmicos, criar estratégias de marketing digital, produzir relatórios ou apoiar decisões, garantindo que as informações sejam sempre atualizadas, precisas e adequadas ao objetivo.',
    tools=[search_tool, scrape_tool],
    verbose=True
)

In [6]:
redator = Agent(
    role='Redator de conteúdo',
    goal='Escreva um texto informal e tecnicamente correto para o LinkedIn sobre {tema}',
    backstory='Como redator de conteúdo, você é responsável por criar textos claros, criativos e envolventes para diferentes plataformas, como blogs, sites, redes sociais e materiais institucionais. Sua função envolve adaptar o tom e o estilo da escrita ao público-alvo, além de transmitir informações de forma estratégica e persuasiva. Você pesquisa, organiza ideias e utiliza técnicas de SEO (quando necessário) para garantir que seu conteúdo seja relevante, acessível e impactante, alinhando sempre sua escrita aos objetivos e valores da marca ou projeto para o qual trabalha.',
    tools=[search_tool, scrape_tool],
    verbose=True
)

In [7]:
editor = Agent(
    role='Editor de conteúdo',
    goal='Editar um texto informal e tecnicamente correto.',
    backstory='Como editor de conteúdo, você é responsável por revisar, refinar e organizar textos, imagens e outros materiais antes de serem publicados. Sua função inclui garantir a qualidade, a clareza e a coerência do conteúdo, além de verificar se ele está alinhado com os objetivos e a identidade da marca. Você ajusta o tom, corrige erros e pode sugerir melhorias para tornar o material mais impactante e envolvente. Além disso, você gerencia prazos e colabora com redatores, designers e outros profissionais para entregar um produto final impecável e relevante para o público-alvo.',
    tools=[search_tool, scrape_tool],
    verbose=True
)

Criação das tarefas

1 - Buscar

2 - Redigir

3 - Editar

In [8]:
buscar = Task(
    description=(
        '1 - Priorize as últimas informações técnicas, os principais atores e as notícias mais relevantes sobre {tema}. \n'
        '2 - Identifique o público-alvo, considerando seus interesses e pontos de dor. \n'
        '3 - Inclua palavras-chave de SEO e dados ou fontes relavantes.'
        ),
    agent=buscador,
    verbose=True,
    expected_output='Um conjunto de informações técnicas relevantes sobre {tema}'
)

In [9]:
redigir = Task(
    description=(
        '1 - Use os dados coletados de conteúdo para criar um post de LinkedIn atraente sobre {tema}. \n'
        '2 - Incorpore palvras-chave de SEO de forma natural. \n'
        '3 - Certifique-se de que o post esteja estruturado de forma cativante, com uma conclusão que faça o leitor refletir e se interessar.'
        ),
    agent=redator,
    verbose=True,
    expected_output='Um texto de LinkedIn sobre {tema}'
)

In [10]:
editar = Task(
    description=(
        'Revisar a postagem do LinkedIn em questão quanto a erros gramaticais e alinhamento com relação ao tema. Considere que estamos no ano 2024 para efeito de avaliação.'
        ),
    agent=editor,
    verbose=True,
    expected_output='Um texto de LinkedIn pronto para publicação.'
)

Criação da equipe e conectar os detalhes

In [11]:
equipe = Crew(
    agents=[buscador, redator, editor],
    tasks=[buscar, redigir, editar],
    verbose=True
)


Execução e resultado

In [12]:
entradas = {
    "tema": 'Recurso WSL do Windows'
}
resultado = equipe.kickoff(inputs=entradas)

# Agent: Buscador de conteúdo na internet
## Task: 1 - Priorize as últimas informações técnicas, os principais atores e as notícias mais relevantes sobre Recurso WSL do Windows. 
2 - Identifique o público-alvo, considerando seus interesses e pontos de dor. 
3 - Inclua palavras-chave de SEO e dados ou fontes relavantes.


# Agent: Buscador de conteúdo na internet
## Thought: Vou começar buscando informações técnicas recentes sobre o Recurso WSL do Windows, focando em notícias, principais atores e atualizações relevantes.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"WSL Windows Subsystem for Linux 2023 news updates features advantages disadvantages\"}"
## Tool Output: 
{'searchParameters': {'q': 'WSL Windows Subsystem for Linux 2023 news updates features advantages disadvantages', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Windows Subsystem for Linux September 2023 update', 'link': 'https://devblogs.microsoft.com/commandline/windo

In [15]:
from IPython.display import Markdown
Markdown(resultado.raw)

🚀 **Explorando o Windows Subsystem for Linux (WSL): A Revolução para Desenvolvedores!** 🐧

Você já se perguntou como é possível ter a flexibilidade do Linux dentro do seu ambiente Windows? O **WSL (Windows Subsystem for Linux)** está aqui para transformar a forma como desenvolvemos software ao integrar duas das plataformas mais poderosas do mundo.

Com a recente atualização de setembro de 2023, o WSL se tornou ainda mais robusto! Agora, desenvolvedores podem tirar proveito de uma **integração aprimorada com o Windows**, melhorias significativas no desempenho e suporte expandido a aplicativos Linux. Essa é uma vitória não apenas para aqueles que trabalham em **projetos Python** ou **DevOps**, mas também para todos que buscam uma transição mais suave entre os ecossistemas Windows e Linux.

### 🔑 **Vantagens do WSL**

1. **Integração Sem Costura:** Execute aplicativos Linux e Windows lado a lado. Imagine desenvolver com suas ferramentas Linux enquanto acessa diretamente arquivos do Windows!

2. **Desempenho Notável:** Com o WSL 2, temos acesso a chamadas de sistema completas, permitindo que aplicativos mais complexos, como **contêineres Docker**, rodem de maneira mais eficiente.

3. **Facilidade de Uso:** A instalação é simples e direta, ideal para quem está mais ansioso para codificar do que para configurar sistemas.

### ⚠️ **Mas e as desvantagens?**

Claro, como tudo na vida, o WSL também tem suas limitações. As **interfaces gráficas** de muitos aplicativos Linux ainda não são totalmente suportadas, e a compatibilidade com alguns **módulos de kernel** pode ser um desafio. Além disso, dependendo do **desempenho do Windows**, o WSL pode não oferecer o máximo desempenho desejado em ambientes de alta demanda.

### 🎯 **Público-Alvo**

O WSL é uma ferramenta direcionada principalmente a **desenvolvedores de software**, **estudantes de ciência da computação** e **profissionais de TI**. Aqueles que buscam uma maneira eficiente de misturar o poder do Linux com a familiaridade do Windows encontrarão o WSL como um aliado indispensável.

### 🌟 **Reflexão Final**

Então, se você é um desenvolvedor que ainda não explorou o WSL, talvez seja hora de reconsiderar. A habilidade de gerenciar e desenvolver projetos de forma híbrida não apenas aumenta a produtividade, mas também supera as barreiras entre diferentes sistemas operacionais.

🔗 Para explorar a atualização mais recente do WSL e descobrir todos os detalhes, confira este [link do blog da Microsoft](https://devblogs.microsoft.com/commandline/windows-subsystem-for-linux-september-2023-update/).

Vamos juntos aprimorar nossa prática de desenvolvimento! Quais são suas experiências com o WSL? Compartilhe nos comentários! 👇

#WindowsSubsystemForLinux #WSL #Desenvolvimento #Python #Inovação #DevOps #Tecnologia #AtualizaçõesWSL